# MyTorch MNIST Lightweight Training (R on Kaggle)
This notebook mirrors `scripts/train_mnist_lightweight_kaggle.R` and is ready for Kaggle R sessions.

In [ ]:
library(keras)
library(jsonlite)
set.seed(23)


In [ ]:
mnist <- dataset_mnist()
x_train <- mnist$train$x
y_train <- mnist$train$y
x_test <- mnist$test$x
y_test <- mnist$test$y


In [ ]:
train_n <- 20000
test_n <- 5000
x_train <- x_train[1:train_n,,,drop=FALSE]
y_train <- y_train[1:train_n]
x_test <- x_test[1:test_n,,,drop=FALSE]
y_test <- y_test[1:test_n]
x_train <- array_reshape(x_train / 255, c(train_n, 784))
x_test <- array_reshape(x_test / 255, c(test_n, 784))
y_train_cat <- to_categorical(y_train, num_classes = 10)
y_test_cat <- to_categorical(y_test, num_classes = 10)


In [ ]:
model <- keras_model_sequential() |>
  layer_dense(units = 256, activation = 'relu', input_shape = c(784)) |>
  layer_dropout(rate = 0.2) |>
  layer_dense(units = 128, activation = 'relu') |>
  layer_dense(units = 10, activation = 'softmax')

model |> compile(
  optimizer = optimizer_adam(learning_rate = 0.001),
  loss = 'categorical_crossentropy',
  metrics = c('accuracy')
)


In [ ]:
history <- model |> fit(
  x_train, y_train_cat,
  epochs = 8,
  batch_size = 128,
  validation_split = 0.1,
  verbose = 2
)


In [ ]:
score <- model |> evaluate(x_test, y_test_cat, verbose = 0)
dir.create('checkpoints', showWarnings = FALSE, recursive = TRUE)
dir.create('outputs', showWarnings = FALSE, recursive = TRUE)
dir.create('../checkpoints', showWarnings = FALSE, recursive = TRUE)
dir.create('../outputs', showWarnings = FALSE, recursive = TRUE)
saveRDS(model, file = 'checkpoints/mnist_lightweight_mlp.rds')
write_json(list(test_loss=unname(score[[1]]), test_accuracy=unname(score[[2]])), 'outputs/metrics.json', auto_unbox=TRUE, pretty=TRUE)
score
